# 3. Fusion, and Handing TE the Whole Block

Notebook 02 ended with a gap. FP8 made the matrix multiplies faster, but a real model spends
time on things FP8 never touches: LayerNorms, activation functions, and the kernel launches
between them. That overhead set a ceiling on the speedup.

This notebook goes after it, by letting Transformer Engine **fuse** neighbouring operations
into whole block kernels.

## Objectives

- Understand why a LayerNorm is expensive despite doing almost no arithmetic
- Replace `LayerNorm + Linear` with `te.LayerNormLinear`, and the MLP with `te.LayerNormMLP`
- Replace the whole block with `te.TransformerLayer`
- Measure each rung, and find the size below which fusion stops paying
- Meet the parameter-naming trap that makes a fused block look worse than it is

## Requirements

- Notebooks 00-02
- The dataset (`make data` from the repository root)

## Working through this on your own

The setup cell repeats the code from notebooks 01 and 02. The measurement cells take about
a minute each.

## 3.0 Setup

Notebooks 01 and 02, restated so this notebook runs standalone.

In [ ]:
import contextlib
import logging
import math
import time
import warnings
from dataclasses import dataclass

logging.getLogger("torch._library.opaque_object").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformer_engine.pytorch as te
from transformer_engine.common import recipe as te_recipe

from helpers import summary, get_tokenizer, get_batch, te_support, warmup
# Import classes built in Notebook 01
from helpers import MiniGPT, TorchBlock, TEUnfusedBlock, train, fp8_context


@dataclass
class Config:
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    seq_len: int = 1024
    vocab_size: int = 50304

def measure(Block, cfg, batch_size, fp8, steps=30, warmup=10):
    """Steady-state training throughput for one block type."""
    time.sleep(10) # cooldown
    torch.manual_seed(1337)
    model = MiniGPT(cfg, Block).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    x, y = get_batch("train", cfg, batch_size)
    def step():
        with torch.autocast("cuda", dtype=torch.bfloat16), fp8_context(fp8):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    for _ in range(warmup):
        step()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(steps):
        step()
    torch.cuda.synchronize()
    tps = x.numel() * steps / (time.perf_counter() - t0)
    del model, opt
    torch.cuda.empty_cache()
    return tps


cfg = Config()
BATCH = 24
print(summary().splitlines()[0])
print(f"config: {cfg.n_layer} layers, n_embd={cfg.n_embd}, "
      f"{BATCH} x {cfg.seq_len} = {BATCH*cfg.seq_len:,} tokens/step")

## 3.1 Why a LayerNorm costs anything

A LayerNorm does almost no arithmetic: for each token it computes a mean and a variance,
then scales. A few operations per number.

But it has to **read every activation from memory and write them all back**, so the next
kernel can read them again. For a batch of 24 x 1024 tokens at width 768, that is about 38
million values moved, twice, per LayerNorm, per layer.

This is the difference between being *compute bound* and *memory bound*. A matrix multiply
does thousands of operations per value it loads, so making the arithmetic faster (FP8) helps
it. A LayerNorm does a handful, so its time is set almost entirely by memory bandwidth --
and FP8 does nothing for it.

**Fusion** is the fix: instead of LayerNorm writing its output for the matmul to read, the
two run as *one kernel* that normalizes values and immediately multiplies them, while they
are still in registers. The round trip through memory disappears.

Transformer Engine provides these fused modules:

| separate | fused |
|---|---|
| `te.LayerNorm` + `te.Linear` | **`te.LayerNormLinear`** |
| `te.LayerNorm` + `te.Linear` + GELU + `te.Linear` | **`te.LayerNormMLP`** |

## 3.2 The fused block

Five modules become two. Everything else -- the attention, the projection, the residual
structure -- is untouched, so any difference we measure is fusion and nothing else.

In [ ]:
class TEFusedBlock(nn.Module):
    """Each LayerNorm folded into the GEMM that follows it."""

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.ln_qkv = te.LayerNormLinear(cfg.n_embd, 3 * cfg.n_embd)   # ln_1 + qkv
        self.attn = te.DotProductAttention(
            cfg.n_head, cfg.n_embd // cfg.n_head, attention_dropout=0.0,
            qkv_format="bshd", attn_mask_type="causal")
        self.proj = te.Linear(cfg.n_embd, cfg.n_embd)
        self.ln_mlp = te.LayerNormMLP(cfg.n_embd, 4 * cfg.n_embd,      # ln_2 + fc_1 + gelu + fc_2
                                      activation="gelu")

    def attention(self, qkv):                    # identical to TEUnfusedBlock
        B, T, C3 = qkv.shape
        C = C3 // 3
        q, k, v = (z.view(B, T, self.n_head, C // self.n_head) for z in qkv.split(C, dim=2))
        return self.proj(self.attn(q, k, v).view(B, T, C))

    def forward(self, x):
        x = x + self.attention(self.ln_qkv(x))   # was: self.qkv(self.ln_1(x))
        return x + self.ln_mlp(x)                # was: fc_2(gelu(fc_1(ln_2(x))))


a = MiniGPT(cfg, TEUnfusedBlock)
b = MiniGPT(cfg, TEFusedBlock)
print(f"unfused: {sum(p.numel() for p in a.parameters())/1e6:.1f}M parameters")
print(f"fused  : {sum(p.numel() for p in b.parameters())/1e6:.1f}M parameters")
print("\nthe fused block, for comparison:\n")
print(b.blocks[0])
del a, b

Same parameter count -- fusion changes *how* the work is scheduled, never what is
computed.

Notice the GELU disappeared from `forward`. It has not gone away; `activation="gelu"` moved
it inside `LayerNormMLP`, so TE applies it without a separate kernel launch and another
round trip through memory. Same saving as the LayerNorm, applied to the activation function.

## 3.3 Measuring fusion

In [ ]:
rows = []
for name, Block in [("TEUnfusedBlock", TEUnfusedBlock), ("TEFusedBlock", TEFusedBlock)]:
    bf16 = measure(Block, cfg, BATCH, fp8=False)
    fp8 = measure(Block, cfg, BATCH, fp8=True)
    rows.append((name, bf16, fp8))
    print(f"  {name:<16}BF16 {bf16:>9,.0f}   FP8 {fp8:>9,.0f}   ({fp8/bf16:.2f}x from FP8)")

print(f"\nfusion alone (BF16):  {rows[1][1]/rows[0][1]:.2f}x")
print(f"fusion + FP8       :  {rows[1][2]/rows[0][1]:.2f}x  vs unfused BF16")

Two different gains, and it is worth separating them.

**Fusion's BF16 gain here is small** -- a couple of percent. That is less than you might
expect from deleting two full round trips through memory per block, and the reason is that
this model spends a lot of its time elsewhere: in the embedding lookup, the attention
kernels, the 50304-wide output GEMM, the loss and the optimizer. The LayerNorms were never a
large share of the total, so removing them cannot buy much.

**But FP8 gains more on the fused block than on the unfused one.** That is the more
interesting number. With less memory traffic in the way, a larger share of what remains is
arithmetic -- and arithmetic is the part FP8 accelerates.

That is the general shape of performance work: **removing one bottleneck increases the
payoff of fixing the next one.** Neither change is impressive alone; together they compound.

## 3.4 Handing TE the whole block

If TE can fuse a LayerNorm into a Linear, why stop there? `te.TransformerLayer` is the
entire block -- attention, MLP, LayerNorms, residual connections -- as one module.

In [ ]:
class TELayerBlock(nn.Module):
    """The whole block, from Transformer Engine."""

    def __init__(self, cfg):
        super().__init__()
        self.layer = te.TransformerLayer(
            cfg.n_embd, 4 * cfg.n_embd, cfg.n_head,
            hidden_dropout=0.0, attention_dropout=0.0,
            self_attn_mask_type="causal",
            # TE defaults to 'sbhd' -- SEQUENCE first. Our tensors are (batch, seq, hidden),
            # so without this it would silently treat the batch axis as the sequence.
            attn_input_format="bshd")

    def forward(self, x):
        return self.layer(x)


m = MiniGPT(cfg, TELayerBlock)
print(f"{sum(p.numel() for p in m.parameters())/1e6:.1f}M parameters\n")
print("parameter names inside one block:")
for n, _ in list(m.blocks[0].named_parameters())[:8]:
    print("  ", n)
del m

## 3.5 The whole ladder

All four block types, same model, same data, same seed.

In [ ]:
LADDER = [("TorchBlock (nb 01)", TorchBlock),
          ("TEUnfusedBlock", TEUnfusedBlock),
          ("TEFusedBlock", TEFusedBlock),
          ("TELayerBlock", TELayerBlock)]

print(f"{'block':<22}{'BF16':>11}{'FP8':>11}{'FP8 gain':>10}{'vs nb 01':>10}")
print("-" * 64)
base = None
for name, Block in LADDER:
    bf16 = measure(Block, cfg, BATCH, fp8=False)
    fp8 = measure(Block, cfg, BATCH, fp8=True) if Block is not TorchBlock else float("nan")
    base = base or bf16
    best = bf16 if Block is TorchBlock else fp8
    gain = "--" if Block is TorchBlock else f"{fp8/bf16:.2f}x"
    print(f"{name:<22}{bf16:>11,.0f}{fp8:>11,.0f}{gain:>10}{best/base:>9.2f}x")

## 3.6 The result depends on how much work you give it

Every number above was measured at 24,576 tokens per step. That was not an arbitrary
choice. Run the same ladder with a smaller batch and the conclusions change.

In [ ]:
print(f"{'tokens/step':>12}{'TorchBlock':>12}{'TEFused+FP8':>13}{'TELayer+FP8':>13}")
print("-" * 50)
for T, B in [(512, 12), (1024, 12), (1024, 24)]:
    c = Config(seq_len=T)
    torch_bf16 = measure(TorchBlock, c, B, fp8=False, steps=30)
    fused = measure(TEFusedBlock, c, B, fp8=True, steps=30)
    layer = measure(TELayerBlock, c, B, fp8=True, steps=30)
    print(f"{B*T:>12,}{1.0:>11.2f}x{fused/torch_bf16:>12.2f}x{layer/torch_bf16:>12.2f}x")

The ordering is the same at every size. The fused block is essentially flat --
it delivers its full gain even at the smallest batch -- while `TransformerLayer` climbs a
little as the batch grows. There is no crossover here and no threshold to worry about: on
this model, each rung of the ladder is worth taking regardless of how much work you hand
the GPU.

That is a more comfortable result than the one in notebook 00, where FP8 *lost* below a
certain width. The difference is that fusion removes memory traffic and kernel launches,
which are costs you pay at any size, whereas FP8 speeds up arithmetic, which only dominates
once the matrices are large enough.

## 3.7 A trap worth meeting: `attn_input_format`

`te.TransformerLayer` takes an argument that decides how it reads your tensor:

```python
attn_input_format = "bshd"     # batch, sequence, hidden  <- what we use
                  = "sbhd"     # sequence, batch, hidden  <- TE's DEFAULT
```

**The default is sequence-first.** Our tensors are `(batch, sequence, hidden)`, so omitting
that argument makes TE read the batch axis as the sequence and vice versa. It does not
raise an error -- 24 and 1024 are both perfectly plausible as either dimension -- so the
model trains, the loss falls, and the throughput number is meaningless.

This is worth dwelling on because it is the hardest class of bug to catch: **the code runs,
the numbers look reasonable, and they are wrong.** During the writing of this workshop that
exact mistake made `TransformerLayer` appear to be the *slowest* block type, which seemed (and was) incorrect.

## 3.8 What you have

The ladder, in one sentence each:

- **`TorchBlock`** -- plain PyTorch. Cannot do FP8.
- **`TEUnfusedBlock`** -- TE modules, one for one. Roughly speed-neutral in BF16, but FP8
  becomes possible.
- **`TEFusedBlock`** -- LayerNorms folded into the following GEMMs. Faster in BF16 *and*
  gets more out of FP8.
- **`TELayerBlock`** -- the whole block from TE. The fastest of the four and the least code
  to write, because TE fuses more than we did by hand (including the bias-add and the
  residual). The cost is that its defaults are its own: see section 3.7.

## 3.9 FP8 outside this workshop

Everything so far used Transformer Engine directly, because that shows the mechanism. In
practice most people meet FP8 through Hugging Face, where it appears under different names
-- and where one distinction causes a lot of confusion.

### Two different things, both called "FP8"

**`transformers` does FP8 for inference.** It appears there as a *quantization method*: you
load an already-trained model with its weights converted to 8 bits, to save memory and run
faster. You cannot train this way.

```python
from transformers import FineGrainedFP8Config, AutoModelForCausalLM

quantization_config = FineGrainedFP8Config()
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B", dtype="auto",
    device_map="auto", quantization_config=quantization_config,
)
```

**`accelerate` does FP8 for training**, and it uses the same Transformer Engine you have
been using all notebook -- just wrapped up:

```python
from accelerate import Accelerator
from accelerate.utils import TERecipeKwargs

accelerator = Accelerator(mixed_precision="fp8", kwarg_handlers=[TERecipeKwargs()])
```

Its `torchao` backend is the other actively maintained option. (A third, MS-AMP, is
deprecated.) Note that neither `transformers` nor `accelerate` is installed in this
container -- the snippets above are for reference, not to run here.

### You already know what its settings mean

Hugging Face's `FineGrainedFP8Config` quantizes **weights** in 2D blocks of 128 x 128 and
**activations** per token in groups of 128. That is exactly the `block` recipe from section
2.7 -- `w_block_scaling_dim=2` for the weight tiles, `x_block_scaling_dim=1` for the
row-wise activations. Same scheme, same origin: it is what enables DeepSeek-V3 support.

The `accelerate` YAML is equally familiar:

```yaml
mixed_precision: fp8
fp8_config:
  backend: TE
  fp8_format: HYBRID        # E4M3 forward, E5M2 gradients
  amax_history_len: 1024    # delayed scaling's window -- section 2.5
  amax_compute_algo: max
```

Every one of those keys is something you have now seen from the inside. That is the payoff
of building it by hand first: the configuration files stop being magic.

### Three practical gotchas

- **`FineGrainedFP8Config` needs compute capability >= 9** (H100 or newer). That excludes
  Ada cards like the L40S and RTX 6000 Ada, which *can* train with FP8 through Transformer
  Engine -- exactly the split you saw in section 2.7.
- **It can silently fall back.** On recent GPUs it dispatches to DeepGEMM kernels, but if
  the CUDA toolkit is missing or too old it logs one warning and quietly uses a slow path.
  Precisely the situation section 2.8 exists for.
- **`torchao` recommends keeping the first and last layers at full precision.** A reasonable
  convention, but worth measuring rather than assuming: notebook 05 finds a case where the
  extra precision boundary costs more than those layers gain.

> **If you take one thing from this section:** "FP8" on a model card usually means
> *quantized for inference*, while "FP8 training" means something quite different and needs
> a different library. Ask which one someone means before comparing numbers with them.


## 3.10 multi-GPU extras from TE

Everything in this workshop runs on one GPU. Real training of anything large does not, and
this is the point where Transformer Engine stops being "layers that do FP8" and starts being
infrastructure can can split work across GPUs.

Transformer Engine [supports four kinds of parallelism](https://docs.nvidia.com/deeplearning/transformer-engine/examples/advanced_optimizations.html#Multi-GPU-training):

### Tensor parallelism -- split each matrix multiply

One layer's weight matrix is divided across GPUs, so each holds a slice and they combine
results. `te.Linear`, `te.LayerNormLinear`, `te.LayerNormMLP` and `te.TransformerLayer` all
take `tp_group`, `tp_size` and `parallel_mode`:

```python
te.Linear(hidden, 3 * hidden, parallel_mode="column", tp_group=group, tp_size=8)
```

`"column"` splits the output dimension, `"row"` the input -- pairing a column-split QKV with
a row-split projection is what lets an attention block cut its communication to one
all-reduce. TE handles that collective for you. Introduced by
[Megatron-LM](https://arxiv.org/abs/1909.08053).

### Sequence parallelism -- split what tensor parallelism leaves behind

Tensor parallelism splits the GEMMs but leaves the LayerNorms and dropouts *replicated* on
every rank: every GPU redundantly normalizes the same activations and stores them. Setting
`sequence_parallel=True` splits those along the sequence dimension instead. It is not a
speedup so much as a memory saving, and it is the difference between fitting a long context
and not. See [Korthikanti et al.](https://arxiv.org/abs/2205.05198).

### Context parallelism -- split the attention itself

Attention cost grows with the *square* of sequence length, so past some length no single GPU
holds one sequence. `te.DotProductAttention` takes `cp_group`, `cp_global_ranks` and
`cp_stream`, and `te.TransformerLayer` has `set_context_parallel_group()`. Ranks exchange
key/value blocks as the attention sweeps across them -- the
[ring-attention](https://arxiv.org/abs/2310.01889) idea. This is what makes 100k-token
contexts trainable.

### Communication overlap -- hide the cost of the above

Tensor parallelism buys speed and pays in collectives. The `ub_*` arguments you may have
noticed in the signatures (`ub_overlap_ag`, `ub_overlap_rs`, `ub_bulk_dgrad`,
`ub_bulk_wgrad`) enable **Userbuffers**, which overlaps those all-gathers and
reduce-scatters with the GEMMs so the network cost hides behind arithmetic that was
happening anyway.

### Where to read more

- [Transformer Engine documentation](https://docs.nvidia.com/deeplearning/transformer-engine/index.html)
- [Source and API](https://github.com/NVIDIA/TransformerEngine) -- the constructor signatures are
  the most reliable reference for which arguments a given version supports
- [Multi-GPU examples](https://github.com/NVIDIA/TransformerEngine/tree/main/examples/pytorch)

Also worth knowing: `te.GroupedLinear` for mixture-of-experts, `te.parallel_cross_entropy`
for a vocabulary-split loss, and `te.checkpoint` for activation recomputation -- the
memory-for-compute trade that usually appears alongside these.

## Exercises

1. **Fuse one half.** Build a block with `te.LayerNormLinear` for the attention but the
   plain `ln_2 + fc_1 + fc_2` MLP. Which half of the fusion is carrying the gain?
2. **Count the kernels.** Use the profiler from notebook 02 section 2.8 to count distinct
   kernels in one forward pass of the unfused and fused blocks. Does the drop match the
   speedup?
3. **Make fusion matter more.** Set `n_embd = 1536` and re-run 3.3. Does fusion's share grow
   or shrink as the matmuls get bigger? Predict before you run.
4. **Break the init deliberately.** Set `RESIDUAL = ("fc_2.weight",)`, train `TEFusedBlock`
   and `TEUnfusedBlock` for 300 steps each, and compare final losses. How large is the
   phantom "fusion hurts quality" effect?
5. **Read TE's choice.** Run with `NVTE_DEBUG=1 NVTE_DEBUG_LEVEL=2` and find which attention
   backend TE selected. Is it the same one `F.scaled_dot_product_attention` uses?
6. **Break it on purpose.** Remove `attn_input_format="bshd"` from `TELayerBlock`, re-run
   section 3.5, and watch the throughput change without any error. Then run the section 3.7
   test against it. How would you have caught this if you had only the speed number?